In [1]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 156 (delta 21), reused 78 (delta 13), pack-reused 70 (from 1)
Receiving objects: 100% (156/156), 121.84 MiB | 30.71 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Updating files: 100% (28/28), done.


In [ ]:
import os
os.chdir("/content/NN-Project1")

In [2]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

2026-05-18 01:02:48.140592: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-18 01:02:48.141089: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 01:02:48.213027: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-18 01:02:49.610139: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [3]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

2026-05-18 01:02:52.820478: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/bluefox/miniconda3/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 44 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [4]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [5]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [6]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [7]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
 33/625 ━━━━━━━━━━━━━━━━━━━━ 11:19 1s/step - accuracy: 0.4781 - f1_score: 0.6945 - loss: 0.7977

KeyboardInterrupt: 

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [ ]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=model.metrics_names)

In [ ]:
model.save("results/models/final_imdb_model.keras")